# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [1]:
import sys
print(sys.executable)

/home/thomas/ai/projects/tinyml-arduino/bin/python


In [2]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"

In [3]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

2026-05-21 00:38:16.278195: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-21 00:38:16.738391: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-21 00:38:16.738443: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-21 00:38:16.741450: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-21 00:38:16.940804: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-21 00:38:16.942835: I tensorflow/core/platform/cpu_feature_guard.cc:182] This Tens

TensorFlow version: 2.14.1
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [5]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [6]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X. <- why this one uppercase and y is lowercase?
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

# <-- Enter your code here <--#
# Separate features and labels
X = df.drop(columns=["Class"])
y = df["Class"]

In [7]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

# <-- Enter your code here <--#
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42
)

In [8]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

# <-- Enter your code here <--#
from sklearn.preprocessing import StandardScaler

# Initialize the scaler
scaler = StandardScaler()

# fit on training data and transform  datasets
X_train_scaled = scaler.fit_transform(X_train) 
X_test_scaled = scaler.transform(X_test)

In [9]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

# <-- Enter your code here <--#
import tensorflow as tf

# One-hot encode the class labels
y_train_encoded = tf.keras.utils.to_categorical(
    y_train,
    num_classes=num_classes
)

y_test_encoded = tf.keras.utils.to_categorical(
    y_test,
    num_classes=num_classes
)

In [10]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

# <-- Enter your code here <--#
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Define the model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')
])

2026-05-21 00:44:41.299029: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-05-21 00:44:41.303857: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2211] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [11]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

# <-- Enter your code here <--#
# Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
history = model.fit(
    X_train_scaled,
    y_train_encoded,
    epochs=20,
    batch_size=8,
    validation_split=0.2
)

Epoch 1/20
13/13 [==============================] - 1s 16ms/step - loss: 1.1149 - accuracy: 0.3939 - val_loss: 0.9734 - val_accuracy: 0.5600
Epoch 2/20
13/13 [==============================] - 0s 4ms/step - loss: 0.8065 - accuracy: 0.8788 - val_loss: 0.7403 - val_accuracy: 0.7600
Epoch 3/20
13/13 [==============================] - 0s 4ms/step - loss: 0.5839 - accuracy: 0.9798 - val_loss: 0.5441 - val_accuracy: 0.8800
Epoch 4/20
13/13 [==============================] - 0s 4ms/step - loss: 0.4105 - accuracy: 0.9798 - val_loss: 0.4111 - val_accuracy: 0.8400
Epoch 5/20
13/13 [==============================] - 0s 5ms/step - loss: 0.2820 - accuracy: 0.9798 - val_loss: 0.3206 - val_accuracy: 0.8800
Epoch 6/20
13/13 [==============================] - 0s 4ms/step - loss: 0.1952 - accuracy: 0.9798 - val_loss: 0.2575 - val_accuracy: 0.8800
Epoch 7/20
13/13 [==============================] - 0s 5ms/step - loss: 0.1425 - accuracy: 0.9798 - val_loss: 0.2098 - val_accuracy: 0.8800
Epoch 8/20
13/13 [=

In [12]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

# <-- Enter your code here <--#
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test_encoded)

# Print test accuracy
print("Test Accuracy:", test_accuracy)

# Make predictions
y_pred_probs = model.predict(X_test_scaled)
y_pred = np.argmax(y_pred_probs, axis=1)

# Print classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Print confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

2/2 [==============================] - 0s 5ms/step - loss: 0.0453 - accuracy: 1.0000
Test Accuracy: 1.0
2/2 [==============================] - 0s 3ms/step

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54


Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


In [28]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

# <-- Enter your code here <--#
# Convert the trained base model to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the model
base_filename = "model_base.tflite"

with open(base_filename, "wb") as f:
    f.write(tflite_model)

# Print file size
print(f"Base TFLite model size: {file_size_kb(base_filename):.2f} KB")

INFO:tensorflow:Assets written to: /tmp/tmp__arpusc/assets


INFO:tensorflow:Assets written to: /tmp/tmp__arpusc/assets


Base TFLite model size: 14.07 KB


2026-05-21 13:12:28.439504: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-21 13:12:28.439899: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-21 13:12:28.442399: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp__arpusc
2026-05-21 13:12:28.443400: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-21 13:12:28.443468: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmp__arpusc
2026-05-21 13:12:28.455374: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-21 13:12:28.523767: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmp__arpusc
2026-05-21 13:12:28.531715: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 89244 m

(a) [Pro] (10 pts) Implement a baseline DNN model for wine classification using the UCI wine dataset.
Report training accuracy, test accuracy, and float32 model size.

 training accuracy: 100%
 
 test accuracy: 100%
 
 float32 model size: 14.07 KB (Im pretty sure base model is float32)

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [14]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

        # (b) Provide representative dataset.
        converter.representative_dataset = (
            lambda: representative_data_gen(X_train_scaled)
        )

        # (c) Set supported ops to INT8.
        converter.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS_INT8
        ]

        # (d) Set input/output types to int8.
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

        # (b) Use float16 quantization.
        converter.target_spec.supported_types = [tf.float16]

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations.
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert and save the model.
    tflite_model = converter.convert()

    with open(filename, "wb") as f:
        f.write(tflite_model)

    # Step 3: Run TFLite inference.
    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    y_pred = []

    for sample in X_test:
        input_data = np.expand_dims(sample, axis=0).astype(np.float32)

        # Quantize input if needed
        if input_details[0]['dtype'] == np.int8:
            input_scale, input_zero_point = input_details[0]['quantization']

            input_data = input_data / input_scale + input_zero_point
            input_data = np.round(input_data).astype(np.int8)

        interpreter.set_tensor(input_details[0]['index'], input_data)
        interpreter.invoke()

        output_data = interpreter.get_tensor(output_details[0]['index'])

        # Dequantize output if needed
        if output_details[0]['dtype'] == np.int8:
            output_scale, output_zero_point = output_details[0]['quantization']

            output_data = (
                output_scale * (output_data.astype(np.float32) - output_zero_point)
            )

        pred_class = np.argmax(output_data, axis=1)[0]
        y_pred.append(pred_class)

    y_pred = np.array(y_pred)
    y_true = np.argmax(y_test_cat, axis=1)

    # Step 4: Report results.
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb(filename):.2f} KB")

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

In [17]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

# <-- Enter your code here <--#
import os
def file_size_kb(filename):
    """Return file size in kilobytes."""
    return os.path.getsize(filename) / 1024
    
# INT8 quantized model
quantize_and_evaluate(
    model,
    X_test_scaled,
    y_test_encoded,
    quant_type='int8',
    filename='model_int8.tflite'
)

# FLOAT16 quantized model
quantize_and_evaluate(
    model,
    X_test_scaled,
    y_test_encoded,
    quant_type='float16',
    filename='model_float16.tflite'
)

# Dynamic range quantized model
quantize_and_evaluate(
    model,
    X_test_scaled,
    y_test_encoded,
    quant_type='dynamic',
    filename='model_dynamic.tflite'
)

INFO:tensorflow:Assets written to: /tmp/tmpqgw95s0z/assets


INFO:tensorflow:Assets written to: /tmp/tmpqgw95s0z/assets
/home/thomas/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-21 00:52:41.147233: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-21 00:52:41.147291: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-21 00:52:41.147494: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpqgw95s0z
2026-05-21 00:52:41.148806: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-21 00:52:41.148822: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpqgw95s0z
2026-05-21 00:52:41.152240: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
20


INT8 TFLite model size: 5.74 KB

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54


Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]
INFO:tensorflow:Assets written to: /tmp/tmpyhx7oq3u/assets


INFO:tensorflow:Assets written to: /tmp/tmpyhx7oq3u/assets
2026-05-21 00:52:41.728184: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-21 00:52:41.728240: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-21 00:52:41.728424: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpyhx7oq3u
2026-05-21 00:52:41.729040: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-21 00:52:41.729096: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpyhx7oq3u
2026-05-21 00:52:41.730891: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-21 00:52:41.767266: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpyhx7oq3u
2026-05-21 00:52:41.777382: I tensorflow/cc/saved_model/loader.cc:316] SavedModel


FLOAT16 TFLite model size: 8.95 KB

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54


Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]
INFO:tensorflow:Assets written to: /tmp/tmp4kkw56oe/assets


INFO:tensorflow:Assets written to: /tmp/tmp4kkw56oe/assets
2026-05-21 00:52:42.309354: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-21 00:52:42.309415: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-21 00:52:42.309645: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp4kkw56oe
2026-05-21 00:52:42.310988: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-21 00:52:42.311005: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmp4kkw56oe
2026-05-21 00:52:42.314583: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-21 00:52:42.344866: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmp4kkw56oe
2026-05-21 00:52:42.352861: I tensorflow/cc/saved_model/loader.cc:316] SavedModel


DYNAMIC TFLite model size: 8.17 KB

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54


Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


(b) [Pro] (20 pts) Quantize the baseline model using:
• Dynamic Range Quantization
• Full Integer Quantization
• Float16 Quantization
Report model size and classification performance (accuracy, classification report, confusion matrix) for
each quantized model.

All statistic above ^^

## Problem 1 - Part (c)

### Pruning

In [18]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

# <-- Enter your code here <--#
import tensorflow_model_optimization as tfmot

# Compute total training steps
batch_size = 8
epochs = 20

end_step = int(np.ceil(len(X_train_scaled) / batch_size) * epochs)

# Define pruning schedule
pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=end_step
)

In [19]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

# <-- Enter your code here <--#
# Alias for convenience
prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

# Build pruned model
pruned_model = tf.keras.Sequential([
    prune_low_magnitude(
        tf.keras.layers.Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
        pruning_schedule=pruning_schedule
    ),

    prune_low_magnitude(
        tf.keras.layers.Dense(32, activation='relu'),
        pruning_schedule=pruning_schedule
    ),

    prune_low_magnitude(
        tf.keras.layers.Dense(3, activation='softmax'),
        pruning_schedule=pruning_schedule
    )
])

In [20]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

# <-- Enter your code here <--#
# Compile the pruned model
pruned_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Define pruning callback
callbacks = [
    tfmot.sparsity.keras.UpdatePruningStep()
]

# Train the pruned model
history_pruned = pruned_model.fit(
    X_train_scaled,
    y_train_encoded,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    callbacks=callbacks
)

Epoch 1/10
13/13 [==============================] - 2s 14ms/step - loss: 1.0733 - accuracy: 0.3636 - val_loss: 0.9758 - val_accuracy: 0.5600
Epoch 2/10
13/13 [==============================] - 0s 5ms/step - loss: 0.7299 - accuracy: 0.8384 - val_loss: 0.6834 - val_accuracy: 0.8000
Epoch 3/10
13/13 [==============================] - 0s 4ms/step - loss: 0.5187 - accuracy: 0.9394 - val_loss: 0.4938 - val_accuracy: 0.9200
Epoch 4/10
13/13 [==============================] - 0s 5ms/step - loss: 0.3700 - accuracy: 0.9596 - val_loss: 0.3620 - val_accuracy: 0.9600
Epoch 5/10
13/13 [==============================] - 0s 5ms/step - loss: 0.2636 - accuracy: 0.9697 - val_loss: 0.2803 - val_accuracy: 0.9600
Epoch 6/10
13/13 [==============================] - 0s 5ms/step - loss: 0.1934 - accuracy: 0.9697 - val_loss: 0.2299 - val_accuracy: 0.9600
Epoch 7/10
13/13 [==============================] - 0s 5ms/step - loss: 0.1496 - accuracy: 0.9697 - val_loss: 0.1984 - val_accuracy: 0.9600
Epoch 8/10
13/13 [=

In [21]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

# <-- Enter your code here <--#
# Remove pruning wrappers
stripped_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(stripped_model)
tflite_pruned_model = converter.convert()

# Save the TFLite model
pruned_filename = "model_pruned.tflite"

with open(pruned_filename, "wb") as f:
    f.write(tflite_pruned_model)

# Print file size
print(f"Pruned TFLite model size: {file_size_kb(pruned_filename):.2f} KB")

INFO:tensorflow:Assets written to: /tmp/tmp98doq7a6/assets


INFO:tensorflow:Assets written to: /tmp/tmp98doq7a6/assets
2026-05-21 01:01:56.728911: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-21 01:01:56.728972: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-21 01:01:56.729187: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp98doq7a6
2026-05-21 01:01:56.729976: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-21 01:01:56.729992: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmp98doq7a6
2026-05-21 01:01:56.732127: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-21 01:01:56.747878: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmp98doq7a6
2026-05-21 01:01:56.753984: I tensorflow/cc/saved_model/loader.cc:316] SavedModel

Pruned TFLite model size: 14.14 KB


In [22]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Generate predictions
y_pred_probs = stripped_model.predict(X_test_scaled)

# Convert probabilities to class labels
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test_encoded, axis=1)

# Print evaluation metrics
print("\nClassification Report:")
print(classification_report(y_true, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

2/2 [==============================] - 0s 5ms/step

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.95      0.97        19
           1       0.95      1.00      0.98        21
           2       1.00      1.00      1.00        14

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54


Confusion Matrix:
[[18  1  0]
 [ 0 21  0]
 [ 0  0 14]]


Pruned TFLite model size: 14.14 KB

## Problem 1 - Part (d)

### Knowledge Distillation

In [23]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

# <-- Enter your code here <--#
# Define the student model
student_model = tf.keras.Sequential([
    tf.keras.layers.Dense(
        32,
        activation='relu',
        input_shape=(X_train_scaled.shape[1],)
    ),
    tf.keras.layers.Dense(
        16,
        activation='relu'
    ),
    tf.keras.layers.Dense(
        3,
        activation='softmax'
    )
])

In [24]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

# <-- Enter your code here <--#
# Generate teacher soft labels
teacher_soft_labels = model.predict(X_train_scaled)

4/4 [==============================] - 0s 3ms/step


In [31]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

# <-- Enter your code here <--#
# Combine hard and soft labels
y_train_combined = np.concatenate(
    [y_train_encoded, teacher_soft_labels],
    axis=1
)

# (b) Define custom distillation loss
def distillation_loss(y_true_combined, y_pred):

    # Split combined labels
    y_true_hard = y_true_combined[:, :3]
    y_true_soft = y_true_combined[:, 3:]

    # Hard-label loss
    hard_loss = tf.keras.losses.categorical_crossentropy(
        y_true_hard,
        y_pred
    ) # i thoughht i liked the other way of writing these in one line but i think i like this awy more

    # Soft-label loss
    soft_loss = tf.keras.losses.categorical_crossentropy(
        y_true_soft,
        y_pred
    )

    # Combine losses
    alpha = 0.5
    return alpha * hard_loss + (1 - alpha) * soft_loss

In [32]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

# <-- Enter your code here <--#
# Compile the student model
student_model.compile(
    optimizer='adam',
    loss=distillation_loss,
    metrics=['accuracy']
)

# Train the student model
history_student = student_model.fit(
    X_train_scaled,
    y_train_combined,
    epochs=10,
    batch_size=8,
    validation_split=0.2
)

Epoch 1/10
13/13 [==============================] - 1s 19ms/step - loss: 1.1818 - accuracy: 0.3535 - val_loss: 1.0855 - val_accuracy: 0.3600
Epoch 2/10
13/13 [==============================] - 0s 4ms/step - loss: 1.0315 - accuracy: 0.4545 - val_loss: 0.9597 - val_accuracy: 0.4800
Epoch 3/10
13/13 [==============================] - 0s 4ms/step - loss: 0.9148 - accuracy: 0.6061 - val_loss: 0.8567 - val_accuracy: 0.6400
Epoch 4/10
13/13 [==============================] - 0s 4ms/step - loss: 0.8110 - accuracy: 0.6970 - val_loss: 0.7620 - val_accuracy: 0.7600
Epoch 5/10
13/13 [==============================] - 0s 5ms/step - loss: 0.7124 - accuracy: 0.8485 - val_loss: 0.6754 - val_accuracy: 0.8400
Epoch 6/10
13/13 [==============================] - 0s 4ms/step - loss: 0.6229 - accuracy: 0.9293 - val_loss: 0.5930 - val_accuracy: 0.9200
Epoch 7/10
13/13 [==============================] - 0s 5ms/step - loss: 0.5392 - accuracy: 0.9293 - val_loss: 0.5167 - val_accuracy: 0.9200
Epoch 8/10
13/13 [=

In [33]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

# <-- Enter your code here <--#
# Convert the student model to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
tflite_kd_model = converter.convert()

# Save the model
kd_filename = "model_kd.tflite"

with open(kd_filename, "wb") as f:
    f.write(tflite_kd_model)

# Print file size
print(f"Knowledge Distillation TFLite model size: {file_size_kb(kd_filename):.2f} KB")

INFO:tensorflow:Assets written to: /tmp/tmpzytmrqa8/assets


INFO:tensorflow:Assets written to: /tmp/tmpzytmrqa8/assets


Knowledge Distillation TFLite model size: 6.10 KB


2026-05-21 13:33:28.613619: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-21 13:33:28.613729: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-21 13:33:28.614668: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpzytmrqa8
2026-05-21 13:33:28.615186: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-21 13:33:28.615199: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpzytmrqa8
2026-05-21 13:33:28.618895: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-21 13:33:28.651510: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpzytmrqa8
2026-05-21 13:33:28.658997: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 44214 m

In [34]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#
# Generate predictions
y_pred_probs = student_model.predict(X_test_scaled)

# Convert probabilities to class labels
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test_encoded, axis=1)

# Print evaluation metrics
print("\nClassification Report:")
print(classification_report(y_true, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

2/2 [==============================] - 0s 3ms/step

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        19
           1       1.00      0.95      0.98        21
           2       1.00      1.00      1.00        14

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54


Confusion Matrix:
[[19  0  0]
 [ 1 20  0]
 [ 0  0 14]]


Knowledge Distillation TFLite model size: 6.10 KB

## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [35]:
# <-- (if needed) Enter your code here <--#
# Convert the student model using INT8 quantization
converter = tf.lite.TFLiteConverter.from_keras_model(student_model)

# Enable optimizations
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Representative dataset for calibration
converter.representative_dataset = (
    lambda: representative_data_gen(X_train_scaled)
)

# Full integer quantization
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]

converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

# Convert model
tflite_kd_int8 = converter.convert()

# Save model
final_filename = "model_kd_int8.tflite"

with open(final_filename, "wb") as f:
    f.write(tflite_kd_int8)

# Print model size
print(f"KD + INT8 model size: {file_size_kb(final_filename):.2f} KB")

INFO:tensorflow:Assets written to: /tmp/tmpi1090h6m/assets


INFO:tensorflow:Assets written to: /tmp/tmpi1090h6m/assets
/home/thomas/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


KD + INT8 model size: 3.62 KB


2026-05-21 13:34:51.444491: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-21 13:34:51.444553: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-21 13:34:51.444878: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpi1090h6m
2026-05-21 13:34:51.445816: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-21 13:34:51.445831: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpi1090h6m
2026-05-21 13:34:51.448647: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-21 13:34:51.477784: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpi1090h6m
2026-05-21 13:34:51.485478: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 40588 m

In [36]:
# Load interpreter
interpreter = tf.lite.Interpreter(model_path=final_filename)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

y_pred = []

for sample in X_test_scaled:

    input_data = np.expand_dims(sample, axis=0).astype(np.float32)

    # Quantize input
    input_scale, input_zero_point = input_details[0]['quantization']

    input_data = input_data / input_scale + input_zero_point
    input_data = np.round(input_data).astype(np.int8)

    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()

    output_data = interpreter.get_tensor(output_details[0]['index'])

    # Dequantize output
    output_scale, output_zero_point = output_details[0]['quantization']

    output_data = (
        output_scale *
        (output_data.astype(np.float32) - output_zero_point)
    )

    pred_class = np.argmax(output_data, axis=1)[0]
    y_pred.append(pred_class)

y_pred = np.array(y_pred)
y_true = np.argmax(y_test_encoded, axis=1)

# Accuracy
accuracy = np.mean(y_pred == y_true)

print("\nAccuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))


Accuracy: 0.9814814814814815

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        19
           1       1.00      0.95      0.98        21
           2       1.00      1.00      1.00        14

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54


Confusion Matrix:
[[19  0  0]
 [ 1 20  0]
 [ 0  0 14]]


The INT8 model had the smallest size, while the original dense model had the best accuracy. The student model from knowledge distillation stayed pretty close in accuracy while using fewer parameters.

To reduce size further, I combined knowledge distillation with INT8 quantization. The smaller student network reduced parameter count, and INT8 quantization compressed the weights even more. This gave the best size reduction without a major drop in classification performance. I reduced the model size by about half while essentially keeping the same accuracy.

At this point, further reduction would probably require removing more neurons or layers, which would start hurting accuracy noticeably since the model is already very small.

# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.


**ChatGPT was used to explain library functions, review strange results, and debug errors.**